# DPG Parameter Sensitivity Benchmark

This notebook analyzes a cross-dataset benchmark for the two main DPG tuning parameters:

- `perc_var`
- `decimal_threshold`

The benchmark results are committed in `tutorials/parameter_sensitivity_benchmark/results/` so practitioners can inspect the parameter behavior without rerunning the full sweep.

Important interpretation note:

- `perc_var` behaves like a graph-pruning control
- `decimal_threshold` depends strongly on the meaningful precision of the variables, so `2` should be read as a starting point rather than a universal default


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'results').exists():
    NOTEBOOK_DIR = Path('tutorials/parameter_sensitivity_benchmark').resolve()

RESULTS_DIR = NOTEBOOK_DIR / 'results'
RESULTS_DIR


In [ ]:
inventory_df = pd.read_csv(RESULTS_DIR / 'dataset_inventory.csv')
summary_df = pd.read_csv(RESULTS_DIR / 'benchmark_summary.csv')
feature_df = pd.read_csv(RESULTS_DIR / 'benchmark_feature_lrc.csv')
stability_df = pd.read_csv(RESULTS_DIR / 'benchmark_stability_vs_baseline.csv')

inventory_df


## Run Overview

In [ ]:
summary_df['status'].value_counts(dropna=False).rename('runs')


In [ ]:
ok_df = summary_df[summary_df['status'] == 'ok'].copy()
ok_df.groupby('dataset', as_index=False).agg(
    successful_runs=('status', 'size'),
    mean_predicates=('n_predicates', 'mean'),
    max_predicates=('n_predicates', 'max'),
    min_predicates=('n_predicates', 'min'),
    mean_accuracy=('test_accuracy', 'mean'),
    mean_runtime=('duration_seconds', 'mean'),
).sort_values(['mean_predicates', 'dataset'], ascending=[False, True])


## Graph Size vs Parameter Choice

In [ ]:
ok_df.groupby(['perc_var', 'decimal_threshold'])['n_predicates'].mean().round(2)


In [ ]:
dataset_name = 'wine'
plot_df = ok_df[ok_df['dataset'] == dataset_name].copy()
pivot_predicates = plot_df.pivot(index='perc_var', columns='decimal_threshold', values='n_predicates').sort_index(ascending=False)
pivot_edges = plot_df.pivot(index='perc_var', columns='decimal_threshold', values='n_edges').sort_index(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pivot_predicates.plot(ax=axes[0], marker='o', title=f'{dataset_name}: predicates by decimal_threshold')
axes[0].set_xlabel('perc_var')
axes[0].set_ylabel('n_predicates')
pivot_edges.plot(ax=axes[1], marker='o', title=f'{dataset_name}: edges by decimal_threshold')
axes[1].set_xlabel('perc_var')
axes[1].set_ylabel('n_edges')
plt.tight_layout()
plt.show()


## Failure Regions

In [ ]:
failure_df = summary_df[summary_df['status'] != 'ok'][['dataset', 'perc_var', 'decimal_threshold', 'error_type', 'error_message']].copy()
failure_df.sort_values(['dataset', 'perc_var', 'decimal_threshold']).reset_index(drop=True)


## Stability vs Baseline

Baseline used in the benchmark:

- `perc_var = 1e-9`
- `decimal_threshold = 2`
- top-10 features by aggregated feature-level LRC


In [ ]:
stability_summary = stability_df.groupby('dataset', as_index=False).agg(
    mean_overlap=('baseline_overlap_ratio', 'mean'),
    min_overlap=('baseline_overlap_ratio', 'min'),
    max_overlap=('baseline_overlap_ratio', 'max'),
).sort_values(['mean_overlap', 'dataset'], ascending=[False, True])
stability_summary


In [ ]:
stability_summary.plot(
    x='dataset',
    y=['mean_overlap', 'min_overlap', 'max_overlap'],
    kind='bar',
    figsize=(14, 4),
    title='Top-10 feature overlap vs baseline',
)
plt.ylabel('overlap ratio')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()


## Plateau Detection

In [ ]:
plateau_rows = []
for dataset, ds in ok_df.groupby('dataset'):
    for perc_var, block in ds.groupby('perc_var'):
        block = block.sort_values('decimal_threshold')
        values = block['n_predicates'].tolist()
        thresholds = block['decimal_threshold'].tolist()
        plateau_dt = None
        for index in range(1, len(values)):
            if values[index] == values[index - 1]:
                plateau_dt = thresholds[index]
                break
        plateau_rows.append(
            {
                'dataset': dataset,
                'perc_var': float(perc_var),
                'plateau_decimal_threshold': plateau_dt,
            }
        )
plateau_df = pd.DataFrame(plateau_rows)
plateau_df.sort_values(['dataset', 'perc_var']).reset_index(drop=True)


## Notes

- The committed notebook is analysis-only.
- The underlying benchmark sweep was run outside the repository and the CSV outputs were copied here.
- The benchmark used a compact Random Forest and capped training rows for large datasets to keep the cross-dataset sweep tractable.
